# Au+Au 200 GeV, 0–10 %: does the O+O chain's energy loss describe central Au+Au?

The calibration test of the tuning page ("A3 and the proposed runs"; `auau_bulk_phobos.ipynb`, "Proposed
runs"). The O+O physics-run chain (`OO_MCGlauber_MUSIC_isr_rhictune.xml`) with Au + Au, b < 4.7 fm (0–10 %),
MUSIC grid ±13 fm, hard partons |y| < 1 (either one), jet in the medium:

| Run | Samples | Energy loss |
| --- | --- | --- |
| B1 | `auau_med_w*` | current: Matter type 1 (2204.01163's Type 2), α_s 0.25, Lbt 0.25 |
| B2 | `auau_t3_w*` | 2204.01163's default Type 3: type 5, α_s 0.3, qhatA 10, qhatB 100, Lbt α_s 0.3 |

Both on the same seeds, 10 p̂_T windows × 2k events, run one after the other (23–80 GeV first, then 9–23,
then 5–9). The notebook runs on whatever has finished (`auau_raa.py`), so it can be re-run during the night.

**Data:**
- **STAR charged jets** (PRC 102, 054913 (2020), arXiv:2006.00582, Fig. 17, read from the vector figure:
  `refs/star_auau200_chjet/`): R_AA^Pythia, anti-k_T R = 0.2, 0.3, 0.4, |η_jet| < 1 − R, leading charged
  track > 5 GeV/c **on the Au+Au jets only: "No p_T,lead cut is imposed on this reference jet population"**
  (Sec. X B). **The p+p reference is PYTHIA (Perugia 2012, PARP(90) 0.213), not a measurement.** At low
  jet p_T the cut removes jets that the reference keeps, which pushes R_AA down: the "biased" points.
  STAR calls the range "~unbiased" above 15 GeV/c (R = 0.2) and 17 GeV/c (R = 0.3, 0.4).
- **JETSCAPE's own Type 3 calculation** (2204.01163 Fig. 14, read from the vector figure:
  `refs/jetscape_2204_fig14/`): the published default (α_s 0.3, Q_sw 2 GeV, τ₀ 0.6) in JETSCAPE's chain
  (TRENTo + free streaming + VISHNU, PP19 p+p, Colorless), the same STAR and PHENIX observables, no retuning
  at RHIC. Purple steps in Figs. 1 and 2.
- **PHENIX π⁰** (PRC 87, 034911 (2013), arXiv:1208.2254, Table VII: `refs/phenix_auau200_pi0/`): |y| < 0.35;
  global uncertainty 9.7 % (p+p normalisation) + 1.5 %.
- **STAR π±** (PRL 108, 072302 (2012), arXiv:1110.0579) and **STAR charged hadrons** (PRL 91, 172302 (2003),
  nucl-ex/0305015), both from HEPData with STAR's measured p+p reference: `refs/star_auau200_hadrons/`.

**Model:** R_AA(p_T) = Σ_AA w dN/dp_T / Σ_pp w dN/dp_T over the hard event's hadrons (one hard scattering
per Au+Au event; w = σ_window / N_window), against `pp_ref` (the same chain in p+p, phase 3 + 3e, with the
same |y| < 1 cut on the hardest partons). The medium's holes and the copies of their colour partners
(iColored hadronizes recoil + partner, the balance is recoil − hole) are subtracted inside the cone: full jets
with the full four-momentum, charged jets with the charged fraction 2/3 of the transverse momentum (≈ 3 GeV
per R = 0.4 jet in central Au+Au, against ≈ 0.1 GeV in O+O). Every jet R_AA is also shown without the
subtraction (thin dotted), to show its size. STAR's leading-track cut is applied as STAR does: to the Au+Au
jets only.
Pions (π⁺ + π⁻)/2 stand in for the π⁰ (decayed in these files). Hadron observables are not hole-subtracted
(`negative_hadrons.py`). The model's hadron R_AA is read from 10 GeV on (grey below): see Fig. 2. Centrality is by b (geometric 0–10 %), the data's by multiplicity.
**Model results, not vetted.**

In [ ]:
import json, subprocess, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
FAM = {"auau_med": ("B1: Type 2, α_s 0.25 (current)", "C3", "o"), "auau_t3": ("B2: Type 3, α_s 0.3", "C0", "s")}
SHOW = ["auau_med"]          # B1 complete (2026-11-26); add "auau_t3" when B2 is complete
FAM = {k: v for k, v in FAM.items() if k in SHOW}
RES = {}
for fam in FAM:
    if list(Path("/data/AuAu-RHIC_phys").glob(f"{fam}_w*/*.json")):
        r = subprocess.run([sys.executable, "auau_raa.py", "--family", fam, "-j", "6"], capture_output=True, text=True)
        print(r.stdout[-2500:] if r.returncode == 0 else r.stderr[-1500:])
        p = Path(f"results/auau_raa_{fam}.json")
        if p.exists():
            RES[fam] = json.loads(p.read_text())
    else:
        print(f"{fam}: no finished jobs yet")
STAR = json.loads(Path("refs/star_auau200_chjet/star_chjet_raa.json").read_text())["centrality"]
PHX = json.loads(Path("refs/phenix_auau200_pi0/phenix_pi0_raa_0_10.json").read_text())["points"]
JS14 = json.loads(Path("refs/jetscape_2204_fig14/js_fig14.json").read_text())["panels"]   # JETSCAPE Type 3, Fig. 14
def js_step(ax_, key, **kw):
    q = JS14[key]
    e = [q[0]["lo"]] + [r["hi"] for r in q]; v = [r["value"] for r in q]
    ax_.stairs(v, e, baseline=None, color="C4", lw=1.8, **kw)
    ax_.errorbar([0.5 * (r["lo"] + r["hi"]) for r in q], v, [r["err_hi"] for r in q], fmt="none", color="C4", lw=1)
def model(fam, key):
    d = RES[fam]["raa"][key]; e = np.array(d["edges"])
    return 0.5 * (e[1:] + e[:-1]), 0.5 * np.diff(e), np.array(d["raa"], float), np.array(d["err"], float)
for fam, r in RES.items():
    print(f"{fam}: {r['auau_events']} Au+Au events in windows {r['auau_windows']}; pp_ref {r['pp_events_in_cut']} events in the cut")

## STAR charged-jet R_AA, 0–10 %

Points with boxes: STAR (filled: "unbiased", open: "biased"; magenta-edged box: correlated, grey: shape
uncertainty; T_AA and PYTHIA normalisation not shown). Model, solid: STAR's definition, the leading-track
cut on Au+Au only (`chjet_R*_star`); dashed: no cut in either; dotted: STAR's definition without the hole
subtraction. Bins with p̂_T windows still missing are not complete; the 23–80 GeV windows run first.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), sharey=True)
for j, R in enumerate((0.2, 0.3, 0.4)):
    for q in STAR["0-10"][f"R{R}"]:
        x, w = 0.5 * (q["lo"] + q["hi"]), 0.5 * (q["hi"] - q["lo"])
        for k, c, al in (("sys_shape", "0.75", 0.6), ("sys_corr", "m", 0.0)):
            if f"{k}_hi" in q:
                ax[j].add_patch(plt.Rectangle((q["lo"], q["value"] - q[f"{k}_lo"]), 2 * w, q[f"{k}_lo"] + q[f"{k}_hi"],
                                              facecolor=c if al else "none", edgecolor=c, alpha=al if al else 1, lw=1))
        ax[j].plot(x, q["value"], "*", ms=10, color="navy", mfc="navy" if q["kind"] == "unbiased" else "white")
    ax[j].plot([], [], "*", ms=10, color="navy", label="STAR 0-10 % (open: biased)")
    js_step(ax[j], f"chjet_R{R}", label="JETSCAPE 2204.01163 Fig. 14: Type 3, α_s 0.3 (PP19 p+p)")
    for fam, (lab, c, mk) in FAM.items():
        if fam not in RES:
            continue
        x, xe, r, e = model(fam, f"chjet_R{R}_star")
        ax[j].errorbar(x, r, e, xe, fmt=mk + "-", ms=4, color=c, label=f"{lab}, p_T,lead > 5 on Au+Au only (STAR)")
        x, xe, r, e = model(fam, f"chjet_R{R}")
        ax[j].plot(x, r, mk + "--", ms=3, mfc="none", color=c, alpha=0.6, label=f"{lab}, no lead cut")
        x, xe, r, e = model(fam, f"chjet_R{R}_star_raw")
        ax[j].plot(x, r, ":", lw=1.2, color=c, label=f"{lab}, STAR's definition, holes not subtracted")
    ax[j].set_title(f"charged jets, R = {R}, |η| < {1 - R:.1f}", fontsize=10); ax[j].set_xlabel(r"$p_{T,jet}^{ch}$ [GeV]")
    ax[j].set_ylim(0, 1.2); ax[j].axhline(1, color="k", lw=0.8); ax[j].set_xlim(5, 32)
ax[0].set_ylabel(r"$R_{AA}$"); ax[0].legend(fontsize=7)
fig.suptitle("Figure 1. Au+Au 0-10 %: charged-jet R_AA, the O+O chain's energy loss against STAR (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
for fam in RES:
    for R in (0.2, 0.3, 0.4):
        x, xe, r, e = model(fam, f"chjet_R{R}_star")
        xr, _, rraw, _ = model(fam, f"chjet_R{R}_star_raw")
        d5 = RES[fam]["raa"][f"chjet_R{R}_lead5"]; d0 = RES[fam]["raa"][f"chjet_R{R}"]
        print(f"{fam} R={R}: p+p fraction of jets with a 5 GeV leading hadron (STAR's PYTHIA 5/0 > 0.9 above 13 / 17 GeV for R = 0.2 / 0.4) "
              + " ".join(f"{xx:.0f}:{a5 / a0:.2f}" for xx, a5, a0 in zip(x, d5["pp"], d0["pp"]) if a0 > 0))
        print(f"{fam} R={R}: R_AA without / with the hole subtraction " + " ".join(f"{xx:.0f}:{a_ / b_:.2f}" for xx, a_, b_ in zip(x, rraw, r) if np.isfinite(a_) and b_ > 0))
        dat = {round(0.5 * (q["lo"] + q["hi"]), 1): q["value"] for q in STAR["0-10"][f"R{R}"]}
        print(f"{fam} R={R}: model/STAR " + " ".join(f"{xx:.0f}:{rr / dat[round(xx, 1)]:.2f}" for xx, rr in zip(x, r) if round(xx, 1) in dat and np.isfinite(rr)))

## Hadron R_AA, 0–10 %: PHENIX π⁰, STAR π±, STAR charged hadrons

- **PHENIX π⁰** (0–10 %, |y| < 0.35): statistical bars, grey boxes type-B systematics; 9.7 % normalisation not shown.
- **STAR π⁺ + π⁻** (PRL 108, 072302 (2012), arXiv:1110.0579, HEPData ins930463; **0–12 %**, |y| < 0.5; measured
  p+p reference): bars statistical, boxes systematic; ≈ 12 % p+p normalisation not shown.
- **STAR (h⁺ + h⁻)/2** (PRL 91, 172302 (2003), nucl-ex/0305015, HEPData ins619063; |η| < 0.5; measured p+p):
  **0–5 %** (filled) and **10–20 %** (open), which bracket our geometric 0–10 %; bars are stat + sys; the N_bin
  normalisation (±16 %) is not shown. `refs/star_auau200_hadrons/fetch_star_hadron_raa.py`.

Model in each experiment's acceptance: pions (π⁺ + π⁻)/2 at |y| < 0.5 (STAR's; solid) and |η| < 0.35 (PHENIX's;
open, faint), charged hadrons |η| < 0.5. **Grey below 10 GeV: the model is statistics-limited there**: hadrons
at 5–10 GeV come mostly from the lowest p̂_T windows (5–11 GeV), where only ≈ 10–50 of 2000 events per window
carry such a hadron (in Au+Au the 5–7 GeV window alone holds half of the 5–8 GeV yield, from 10 events). Medium
response (hadrons are not hole-subtracted) adds to it at the low end. Right panel: model only.

In [ ]:
SH = json.loads(Path("refs/star_auau200_hadrons/star_hadron_raa.json").read_text())
def integ(fam, key, lo, hi):
    d = RES[fam]["raa"][key]; e = np.array(d["edges"]); a = np.array(d["auau"]); p = np.array(d["pp"]); er = np.array(d["err"])
    s = (e[:-1] >= lo - 1e-9) & (e[1:] <= hi + 1e-9)
    return a[s].sum() / p[s].sum(), np.sqrt(((er * p)[s] ** 2).sum()) / p[s].sum()
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
for q in PHX:
    ax[0].add_patch(plt.Rectangle((q["lo"], q["value"] - q["syst_B"]), q["hi"] - q["lo"], 2 * q["syst_B"], color="0.6", alpha=0.35, lw=0))
ax[0].errorbar([q["pt"] for q in PHX], [q["value"] for q in PHX], [q["stat"] for q in PHX], fmt="o", ms=4, color="k", label="PHENIX π⁰ 0-10 %, |y| < 0.35")
for q in SH["pion"]["points"]:
    if q["hi"] <= 4.5:
        continue
    ax[0].add_patch(plt.Rectangle((q["lo"], q["value"] - q["syst"]), q["hi"] - q["lo"], 2 * q["syst"], facecolor="none", edgecolor="navy", lw=0.8))
    ax[0].errorbar(0.5 * (q["lo"] + q["hi"]), q["value"], q["stat"], fmt="*", ms=9, color="navy")
ax[0].plot([], [], "*", ms=9, color="navy", label="STAR π± 0-12 %, |y| < 0.5")
js_step(ax[0], "pion", label="JETSCAPE 2204.01163 Fig. 14: Type 3, (π⁺+π⁻)/2 |y| < 0.35")
for c_, (ps, mfc) in {"0-5%": ("navy", "navy"), "10-20%": ("navy", "white")}.items():
    pts = [q for q in SH["hadron"]["centrality"][c_] if q["hi"] > 4.5]
    ax[1].errorbar([0.5 * (q["lo"] + q["hi"]) for q in pts], [q["value"] for q in pts], [q["stat_syst"] for q in pts],
                   [0.5 * (q["hi"] - q["lo"]) for q in pts], fmt="D", ms=5, color=ps, mfc=mfc, label=f"STAR (h⁺+h⁻)/2 {c_}, |η| < 0.5")
for fam, (lab, c, mk) in FAM.items():
    if fam not in RES:
        continue
    x, xe, r, e = model(fam, "pion_star")
    ax[0].errorbar(x, r, e, xe, fmt=mk, ms=4, color=c, label=f"{lab}: π±, |y| < 0.5")
    x, xe, r, e = model(fam, "pion")
    ax[0].errorbar(x, r, e, xe, fmt=mk, ms=3, mfc="none", color=c, alpha=0.35, label=f"{lab}: π±, |η| < 0.35")
    x, xe, r, e = model(fam, "had_star")
    ax[1].errorbar(x, r, e, xe, fmt=mk + "-", ms=4, color=c, label=f"{lab}: (h⁺+h⁻)/2, |η| < 0.5")
    x, xe, r, e = model(fam, "had")
    ax[2].errorbar(x, r, e, xe, fmt=mk + "-", ms=4, color=c, label=f"{lab}: charged hadrons |η| < 1")
    x, xe, r, e = model(fam, "fjet_R0.4")
    ax[2].errorbar(x, r, e, xe, fmt=mk + "--", ms=4, mfc="none", color=c, label=f"{lab}: full jets R = 0.4, |η| < 0.5")
    x, xe, r, e = model(fam, "fjet_R0.4_raw")
    ax[2].plot(x, r, ":", lw=1.2, color=c, label=f"{lab}: full jets, holes not subtracted")
for a_ in ax:
    a_.axhline(1, color="k", lw=0.8); a_.set_ylim(0, 1.6); a_.set_xlabel(r"$p_T$ [GeV]"); a_.legend(fontsize=7, loc="upper right")
for a_ in ax[:2]:
    a_.axvspan(0, 10, color="0.92", zorder=0)
ax[0].set_xlim(4.5, 20.5); ax[1].set_xlim(4.5, 15.5); ax[2].set_xlim(4.5, 50)
ax[0].set_ylabel(r"$R_{AA}$")
ax[0].set_title("pions: PHENIX π⁰, STAR π±; grey: model statistics-limited", fontsize=10)
ax[1].set_title("charged hadrons: STAR 0-5 % and 10-20 % bracket 0-10 %", fontsize=10)
ax[2].set_title("model only: charged hadrons and full jets", fontsize=10)
fig.suptitle("Figure 2. Au+Au 0-10 %: hadron R_AA against PHENIX and STAR, and the model's full-jet R_AA (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
st_pi = [q for q in SH["pion"]["points"] if q["lo"] >= 10]
st_h = {c_: [q for q in SH["hadron"]["centrality"][c_] if q["lo"] >= 8] for c_ in ("0-5%", "10-20%")}
print("data 10-15 GeV: STAR pi+- " + " ".join(f"{q['lo']:g}-{q['hi']:g}:{q['value']:.2f}" for q in st_pi)
      + " | PHENIX pi0 " + " ".join(f"{q['lo']:g}-{q['hi']:g}:{q['value']:.2f}" for q in PHX if 10 <= q["lo"] < 15)
      + " | STAR h 8-10 GeV " + "; ".join(f"{c_} " + " ".join(f"{q['value']:.2f}" for q in v) for c_, v in st_h.items()))
for fam in RES:
    for key, lo, hi in (("pion_star", 10, 15), ("pion", 10, 20), ("had_star", 10, 15), ("had", 10, 25)):
        r_, e_ = integ(fam, key, lo, hi)
        print(f"{fam}: {key} {lo}-{hi} GeV (integrated): R_AA {r_:.2f} ± {e_:.2f}")

## The p+p reference: data, Pythia 8 and JETSCAPE PP19

**What `pp_ref` is.** The full chain in p+p (`pp200_isr_rhictune.xml`, b < 3 fm): 3dMCGlauber strings, MUSIC
and iSS where the event has a string (29 % of the jet events; 71 % have no medium at all), PythiaIsrGun +
iMATTER, the hard-momentum subtraction and string veto as in O+O, iColored. **The jet stays in vacuum**
(`vacuum_jet 1`: Matter `in_vac 1` showers to Q0 = 2 GeV, no LBT, no recoils), so the hydro does not touch
the jet observables; the bulk is in a separate file and not used here. The Au+Au medium arm showers the
vacuum part to QS = 0.9 GeV instead (A+A scheme); that changes jets by < 1 % and leading hadrons by at most
a few % (tuning page, "Phase 4"). The hard tune fitted α_s(M_Z) = 0.153 to STAR's p+p full jets.

**Comparison** (`pp_ref_check.py`, the same code for all samples, hard event only): `pp_ref` (70k events),
Pythia 8.312 default (Monash) and STAR's Detroit tune (`refs/pythia_ref.py`, 160k events each, the same
p̂_T windows, /data/OO-RHIC_pp_refs) and JETSCAPE's PP19 (arXiv:1910.05481, the p+p baseline of 2204.01163:
Pythia's ISR and MPI, MATTER in vacuum to Q0 = 1 GeV, Colorless; `refs/pp19_200.xml`,
`refs/hadronize_pp19_icdecays.xml`, 160k events), all **with iColored's decays**: iColored (X-SCAPE
`iColoredHadronization.cc`, no `<weak_decays>` in our XML) sets `HadronLevel:Decay on`,
`ParticleDecays:limitTau0 on`, `tau0Max = 10 mm`, so π⁰, η and all resonances decay and K⁰_S, Λ, Σ±, Ξ, Ω do
not. That is the data's "primary" definition: STAR subtracts the weak-decay feed-down (K⁰_S → π⁺π⁻) from its
π± and corrects its h± for Λ, K⁰_S, Σ daughters; its charged jets use primary tracks (DCA < 1 cm). The Pythia
samples get the same four lines (`--line`; `*_icdecays`), checked by their species content (K⁰_S 2.10 %,
Λ 0.69 %, e± 0.62 %, no π⁰/η, against `pp_ref`'s 2.14 / 0.74 / 0.62 %). `pythia_ref.py`'s own default (all
decays except π⁰, η) adds the K⁰_S and Λ daughters to the charged jets: + 5–18 % at R = 0.4. Data: STAR π± and (h⁺+h⁻)/2 p+p (the references of the STAR R_AA above;
NSD yields × 30 mb), PHENIX π⁰ (`data/data.json`), STAR full jets R = 0.5 (`pp_jet_spectra.py`).
**STAR's charged-jet R_AA uses PYTHIA 6 Perugia 2012 with PARP(90) = 0.213 as p+p** (2006.00582 Sec. IX):
not available here; Monash and Detroit stand in for it. Pythia's jets include its underlying event;
"UE-subtracted" removes the mean charged p_T in two perpendicular cones (≈ 0.2 GeV per R = 0.4 jet in Pythia,
0.1 GeV in `pp_ref`). The p̂_T windows start at 5 GeV, so hadrons are compared from 6 GeV.

In [ ]:
if not Path("results/pp_ref_check.json").exists():
    subprocess.run([sys.executable, "pp_ref_check.py", "-j", "6"], check=True, capture_output=True)
PR = json.loads(Path("results/pp_ref_check.json").read_text())["samples"]
DD = json.loads(Path("data/data.json").read_text())
PJ = json.loads(Path("results/pp_jet_spectra.json").read_text())
PPS = {"pp_ref": ("pp_ref (our chain)", "k", "o"), "pythia8_default_ic": ("Pythia 8 Monash", "C1", "s"),
       "pythia8_detroit_ic": ("Pythia 8 Detroit", "C2", "^"),
       "pp19_ic": ("JETSCAPE PP19", "C4", "D")}
def pdata(k):
    if k == "pi_star":
        return {(q["lo"], q["hi"]): (q["value"] * 30, np.hypot(q["stat"], q["syst"]) * 30) for q in SH["pp_pion"]["points"]}
    if k == "h_star":
        return {(q["lo"], q["hi"]): (q["value"] * 30, q["stat_syst"] * 30) for q in SH["pp_hadron"]["points"]}
    P_ = DD["phenix_pi0"]
    return {tuple(e): (v, np.hypot(a, b)) for e, v, a, b in zip(P_["edges"], P_["value"], P_["stat"], P_["sys"])}
def ratio_to_data(n, k, lo=6):
    s = PR[n]["spectra"][k]; e = s["edges"]; dd = pdata(k); x, xe, r, er, de = [], [], [], [], []
    for i in range(len(e) - 1):
        if e[i] < lo or (e[i], e[i + 1]) not in dd:
            continue
        v, dv = dd[(e[i], e[i + 1])]
        x.append(0.5 * (e[i] + e[i + 1])); xe.append(0.5 * (e[i + 1] - e[i])); r.append(s["value"][i] / v); er.append(s["err"][i] / v); de.append(dv / v)
    return map(np.array, (x, xe, r, er, de))
fig, ax = plt.subplots(1, 4, figsize=(21, 4.4))
for j_, (k, tit, lo) in enumerate((("pi_star", "π±, |y| < 0.5: model / STAR p+p", 6), ("pi_phenix", "π (model) / PHENIX π⁰ p+p, |y| < 0.35", 6),
                                    ("h_star", "(h⁺+h⁻)/2, |η| < 0.5: model / STAR p+p", 6))):
    for n, (lab, c, mk) in PPS.items():
        x, xe, r, er, de = ratio_to_data(n, k, lo)
        ax[j_].errorbar(x + {"pp_ref": 0, "pythia8_default_ic": 0.12, "pythia8_detroit_ic": -0.12, "pp19_ic": 0.24}[n], r, er, fmt=mk, ms=4, color=c, label=lab)
    for xx, xw, d_ in zip(x, xe, de):
        ax[j_].add_patch(plt.Rectangle((xx - xw, 1 - d_), 2 * xw, 2 * d_, color="0.85", lw=0, zorder=0))
    ax[j_].set_title(tit, fontsize=10); ax[j_].set_ylim(0, 2.5); ax[j_].axhline(1, color="k", lw=0.8); ax[j_].set_xlabel(r"$p_T$ [GeV]")
ax[0].set_ylabel("model / data"); ax[0].legend(fontsize=7)
for Rj, ls in ((0.2, ":"), (0.3, "--"), (0.4, "-")):
    e = np.array(PR["pp_ref"]["spectra"][f"chjet_R{Rj}_uesub"]["edges"]); xc = 0.5 * (e[1:] + e[:-1])
    a_ = np.array(PR["pp_ref"]["spectra"][f"chjet_R{Rj}_uesub"]["value"])
    for n, c in (("pythia8_detroit_ic", "C2"), ("pythia8_default_ic", "C1"), ("pp19_ic", "C4")):
        ax[3].plot(xc, a_ / np.array(PR[n]["spectra"][f"chjet_R{Rj}_uesub"]["value"]), ls, color=c, marker="o", ms=3,
                   label=f"R = {Rj}: pp_ref / {PPS[n][0].split()[-1]}")
st = DD["star_jets_eta0"]; ed = np.array(st["edges"]); m_ = np.array(PJ["star_eta0"]["value"]); dv = np.array(st["value"])
ax[3].plot(ed.mean(1), m_ / dv, "k*", ms=8, label="full jets R = 0.5, |η| < 0.5: pp_ref / STAR p+p")
ax[3].set_ylim(0, 1.6); ax[3].set_xlim(7, 32); ax[3].axhline(1, color="k", lw=0.8); ax[3].legend(fontsize=6.5, loc="lower left")
ax[3].set_title("charged jets (UE-subtracted): pp_ref / Pythia 8, PP19", fontsize=10); ax[3].set_xlabel(r"$p_{T,jet}$ [GeV]")
fig.suptitle("Figure 4. The p+p reference: pp_ref, Pythia 8 and JETSCAPE PP19 against p+p data, and pp_ref's charged jets against them (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
for k, lo, hi in (("pi_star", 6, 15), ("pi_phenix", 6, 20), ("h_star", 6, 10)):
    row = []
    for n in PPS:
        x, xe, r, er, de = ratio_to_data(n, k)
        s_ = (x - xe >= lo) & (x + xe <= hi); w_ = 1 / er[s_] ** 2
        row.append(f"{n} {np.sum(r[s_] * w_) / w_.sum():.2f} ± {w_.sum() ** -0.5:.2f}")
    print(f"{k} {lo}-{hi} GeV, model/data (weighted): " + " | ".join(row))
for Rj in (0.2, 0.3, 0.4):
    a_ = np.array(PR["pp_ref"]["spectra"][f"chjet_R{Rj}_uesub"]["value"])
    print(f"charged jets R = {Rj} (UE-subtracted), pp_ref / Detroit " + " ".join(f"{v:.2f}" for v in a_ / np.array(PR["pythia8_detroit_ic"]["spectra"][f"chjet_R{Rj}_uesub"]["value"]))
          + " | / Monash " + " ".join(f"{v:.2f}" for v in a_ / np.array(PR["pythia8_default_ic"]["spectra"][f"chjet_R{Rj}_uesub"]["value"]))
          + " | / PP19 " + " ".join(f"{v:.2f}" for v in a_ / np.array(PR["pp19_ic"]["spectra"][f"chjet_R{Rj}_uesub"]["value"])))

### What the reference does to the jet R_AA

STAR divides by PYTHIA, we divide by `pp_ref`. Figure 5 divides the model's Au+Au charged jets (STAR's
definition: leading-track cut on Au+Au only) by each p+p sample: solid `pp_ref` (the R_AA of Fig. 1), dashed
Pythia 8 Detroit, dotted Monash, dash-dotted JETSCAPE PP19 (2204.01163's p+p baseline), all without the leading-track cut and with their own underlying event, as
STAR's PYTHIA reference. This emulates STAR's construction with a PYTHIA 8 stand-in for PYTHIA 6.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4), sharey=True)
for j, Rj in enumerate((0.2, 0.3, 0.4)):
    for q in STAR["0-10"][f"R{Rj}"]:
        ax[j].add_patch(plt.Rectangle((q["lo"], q["value"] - q.get("sys_corr_lo", 0)), q["hi"] - q["lo"],
                                      q.get("sys_corr_lo", 0) + q.get("sys_corr_hi", 0), facecolor="none", edgecolor="m", lw=1))
        ax[j].plot(0.5 * (q["lo"] + q["hi"]), q["value"], "*", ms=10, color="navy", mfc="navy" if q["kind"] == "unbiased" else "white")
    for fam, (lab, c, mk) in FAM.items():
        d = RES[fam]["raa"][f"chjet_R{Rj}_lead5"]; e = np.array(d["edges"]); xc = 0.5 * (e[1:] + e[:-1])
        A_ = np.array(d["auau"]) / np.diff(e) / (2 * (1 - Rj))
        for n, ls, cc in (("pp_ref", "-", c), ("pythia8_detroit_ic", "--", c), ("pythia8_default_ic", ":", c), ("pp19_ic", "-.", "C4")):
            pp_ = np.array(PR[n]["spectra"][f"chjet_R{Rj}"]["value"])
            ax[j].plot(xc, A_ / pp_, ls, color=cc, marker=mk if n == "pp_ref" else None, ms=4, label=f"{lab} / {PPS[n][0]}")
        print(f"{fam} R = {Rj}: Au+Au / pp_ref " + " ".join(f"{v:.2f}" for v in A_ / np.array(PR['pp_ref']['spectra'][f'chjet_R{Rj}']['value']))
              + " | / Detroit " + " ".join(f"{v:.2f}" for v in A_ / np.array(PR['pythia8_detroit_ic']['spectra'][f'chjet_R{Rj}']['value']))
              + " | / Monash " + " ".join(f"{v:.2f}" for v in A_ / np.array(PR['pythia8_default_ic']['spectra'][f'chjet_R{Rj}']['value']))
              + " | / PP19 " + " ".join(f"{v:.2f}" for v in A_ / np.array(PR['pp19_ic']['spectra'][f'chjet_R{Rj}']['value'])))
    ax[j].set_title(f"charged jets R = {Rj}, STAR's definition", fontsize=10); ax[j].set_xlabel(r"$p_{T,jet}^{ch}$ [GeV]")
    ax[j].set_ylim(0, 1.0); ax[j].set_xlim(7, 31)
ax[0].plot([], [], "*", color="navy", label="STAR 0-10 % (open: biased)"); ax[0].legend(fontsize=7); ax[0].set_ylabel(r"$R_{AA}$")
fig.suptitle("Figure 5. The model's charged-jet R_AA with four p+p references: pp_ref, Pythia 8 (as STAR) and JETSCAPE PP19 (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()

## The medium at mid-rapidity, 0–10 %, and the per-parton jet loss

The bulk check that rides along (`auau_bulk_phobos.ipynb`): charged dN/dη at |η| < 0.5 (PHOBOS 0–10 % ≈ 609,
the average of its 0–3, 3–6, 6–10 % bins at |η| < 1) and ⟨p_T⟩ of π, K, p at mid-rapidity (iSS, no
afterburner). Then the per-parton jet loss of `quench_diag.ipynb` in Au+Au, against `oo_vac` as the vacuum
reference (the jet's hadrons only, so the vacuum ratio hardly depends on the system), next to central O+O.

In [ ]:
for fam, r in RES.items():
    if "bulk_0_10" in r:
        b = r["bulk_0_10"]
        print(f"{fam}: {b['events']} events: dN_ch/deta |eta|<0.5 = {b['dnch_deta_mid']:.0f} (PHOBOS 0-10 % ≈ 609 at |eta|<1); "
              f"<pT> pi {b['mean_pt_pi']:.3f}, K {b['mean_pt_K']:.3f}, p {b['mean_pt_p']:.3f} GeV")
import glob, os
from concurrent.futures import ProcessPoolExecutor
import quench_diag as Q
D = np.load("results/quench_diag_eloss.npz")
b20 = np.quantile(D["mb_b"], 0.2)
evv = D["oo_vac/ev"].astype(int); mv0 = D["oo_vac/ev_b"][evv] < b20
evm = D["oo_med/ev"].astype(int); mm0 = D["oo_med/ev_b"][evm] < b20
PT = [10, 15, 20, 25, 30, 40, 55, 80]
def wmean(x, w):
    ok = np.isfinite(x); return (w[ok] * x[ok]).sum() / w[ok].sum()
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
xc = 0.5 * (np.array(PT[1:]) + np.array(PT[:-1]))
def vac_ref(lo, hi):
    v = mv0 & (D["oo_vac/pt"] >= lo) & (D["oo_vac/pt"] < hi)
    return wmean(D["oo_vac/r_had"][v], D["oo_vac/ev_w"][evv][v])
oo_loss = [1 - wmean(D["oo_med/r_had"][mm0 & (D["oo_med/pt"] >= lo) & (D["oo_med/pt"] < hi)],
                     D["oo_med/ev_w"][evm][mm0 & (D["oo_med/pt"] >= lo) & (D["oo_med/pt"] < hi)]) / vac_ref(lo, hi) for lo, hi in zip(PT[:-1], PT[1:])]
oo_med = [wmean((D["oo_med/e_med_hemi"] / D["oo_med/E"])[mm0 & (D["oo_med/pt"] >= lo) & (D["oo_med/pt"] < hi)],
                D["oo_med/ev_w"][evm][mm0 & (D["oo_med/pt"] >= lo) & (D["oo_med/pt"] < hi)]) for lo, hi in zip(PT[:-1], PT[1:])]
ax[0].plot(xc, 100 * np.array(oo_med), "k^--", label="O+O 0-20 % (b), current setting")
ax[1].plot(xc, 100 * np.array(oo_loss), "k^--", label="O+O 0-20 % (b), current setting")
for fam, (lab, c, mk) in FAM.items():
    fs = sorted(f for f in glob.glob(f"/data/AuAu-RHIC_phys/{fam}_w*/*_jet_frag.h5") if os.path.exists(f.replace("_hadrons_jet_frag.h5", ".json")))
    if not fs:
        continue
    with ProcessPoolExecutor(max_workers=6) as ex:
        rr = list(ex.map(Q.eloss_file, fs))
    pt = np.concatenate([o["pt"] for *_, o in rr]); E = np.concatenate([o["E"] for *_, o in rr])
    rh = np.concatenate([o["r_had"] for *_, o in rr]); eh = np.concatenate([o["e_med_hemi"] for *_, o in rr])
    # weight: sigma / N per window (one window per file here)
    w = np.concatenate([np.full(len(o["pt"]), s[0] / max(na[0], 1)) for _, s, na, n, ev, o in rr])
    loss, med = [], []
    for lo, hi in zip(PT[:-1], PT[1:]):
        m = (pt >= lo) & (pt < hi)
        loss.append(1 - wmean(rh[m], w[m]) / vac_ref(lo, hi) if m.sum() > 30 else np.nan)
        med.append(wmean((eh / E)[m], w[m]) if m.sum() > 30 else np.nan)
    ax[0].plot(xc, 100 * np.array(med), mk + "-", color=c, label=f"Au+Au 0-10 %, {lab}")
    ax[1].plot(xc, 100 * np.array(loss), mk + "-", color=c, label=f"Au+Au 0-10 %, {lab}")
    print(f"{fam}: {len(pt)} partons; energy to the medium [%] " + " ".join(f"{x:.1f}" for x in 100 * np.array(med))
          + " | jet loss (hadron) [%] " + " ".join(f"{x:.1f}" for x in 100 * np.array(loss)))
ax[0].set_title("energy given to the medium, hemisphere [% of the parton's E]", fontsize=9)
ax[1].set_title("jet loss 1 - <r>_med/<r>_vac (hadron level, R = 0.4), vacuum: oo_vac", fontsize=9)
for a_ in ax:
    a_.set_xlabel("parton $p_T$ [GeV]"); a_.legend(fontsize=7); a_.set_ylim(0, None)
fig.suptitle("Figure 3. Per-parton energy loss: central Au+Au against central O+O (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()

## Reading (B1 complete, 2026-11-26; B2 running)

B1 = the current O+O setting (Matter type 1 = 2204.01163's Type 2, α_s 0.25, Lbt α_s 0.25, Q0 2, checked
in the job logs), 19,999 Au+Au events at b < 4.7 fm, all 10 windows; p+p: 66k `pp_ref` events in the cut.
Averages below are inverse-variance weighted, model statistical errors only. **Model results, not vetted.**

1. **B1 under-quenches central Au+Au: R_AA ≈ 1.3–2× the data, for hadrons and jets.**
   - Hadrons, 10–15 GeV (integrated; the model is statistics-limited below 10 GeV): π± at |y| < 0.5
     0.45 ± 0.10 against STAR π± 0.31–0.32 (0–12 %) and PHENIX π⁰ 0.23–0.24 (10–14 GeV). Model/data
     ≈ 1.4 (STAR π±) to ≈ 1.9 (PHENIX π⁰), each ± ≈ 0.3 from the model's statistics.
   - STAR's charged hadrons stop at 10 GeV, where the model becomes usable, so they do not test it directly:
     the model's 0.37 ± 0.07 at 10–15 GeV sits above STAR's last points (8–10 GeV: 0.19–0.22 at 0–5 %,
     0.25–0.32 at 10–20 %), with R_AA nearly flat in p_T in the data.
   - STAR's π± lie ≈ 0.07 (≈ 30 %) above PHENIX's π⁰; that is within their combined systematic and
     normalisation uncertainties (STAR ± 0.04–0.05 and 12 %, PHENIX ± 0.03 and 9.7 %).
   - Charged jets with STAR's definition (the 5 GeV leading-track cut on Au+Au only), in STAR's
     "~unbiased" range 16–30 GeV: 0.46–0.57 for all three R. STAR: 0.25–0.38. Model/STAR ≈ 2.1–2.3 at
     16–20 GeV, 1.2–1.5 at 20–30 GeV, beyond STAR's correlated + shape systematics.
   - Hadrons and jets are both off by about the same factor: the overall strength is too low, not the shape
     of the jet or the cone.
2. **The leading-track cut matters in the biased region, and only there.**
   - STAR's PYTHIA reference has no cut, so at low jet p_T the cut removes p+p jets too and lowers R_AA.
     With STAR's definition the model drops at 8–14 GeV by 15–55 % (R = 0.4, 8–10 GeV: 0.87 with the cut in
     both → 0.38), and now rises with p_T (R = 0.4: 0.38 → ≈ 0.55), as STAR does. It is still 2–3× STAR there.
   - Above 15–17 GeV the definitions agree to ≤ 4 %.
   - Our cut reproduces STAR's PYTHIA bias: in `pp_ref` the fraction of jets keeping a 5 GeV leading hadron
     passes 0.9 at 12–14 GeV (R = 0.2) and 16–20 GeV (R = 0.4); STAR's Fig. 13: 13 and 17 GeV.
   - The model's R dependence in the unbiased range is weak, like STAR's; B2 will tell more.
3. **Medium response is large in central Au+Au.** Without the hole + partner subtraction, R = 0.4 charged
   jets are 1.2–1.36× higher and full R = 0.4 jets go from 0.51 to 0.77 (Fig. 2, dotted): about a third of
   the energy lost from an R = 0.4 cone comes back as recoils. In O+O this is negligible. The R = 0.2 jets
   change by only 3–9 %.
4. **The model's hadron R_AA below 10 GeV is limited by statistics, not physics.** At 5–10 GeV it scatters
   around 1 with ± 0.3–0.9 per bin. Those hadrons come mostly from the lowest p̂_T windows (5–11 GeV), where only
   ≈ 10–50 of the 2000 Au+Au events per window carry one (a hadron near the parton's p_T is rare); in Au+Au the
   5–7 GeV window alone holds half of the 5–8 GeV yield, from 10 events. Medium response (hadrons are not
   hole-subtracted) may add at the low end, but cannot be separated with these statistics. Read hadrons from
   10 GeV on. A test of hadron R_AA below 10 GeV needs many more events at p̂_T 5–15 GeV (cheap: a hadron-level
   p+p and Au+Au sample with more events in the low windows), not more of the high windows.
5. **Per parton (Fig. 3), Au+Au is much stronger than O+O, but still not enough.** A 30–40 GeV parton
   gives 12 % of its energy to the medium (O+O 0–20 %: 5 %), and its R = 0.4 jet loses 18 % at hadron level
   (O+O: 2.4 %). At 10–15 GeV: 27 % to the medium, 37 % jet loss. The loss falls steeply with p_T.
   The average loss and R_AA are not the same thing: event-by-event fluctuations (short paths, partons
   near the surface) keep R_AA higher than a mean-loss estimate would.
6. **The bulk is fine for this test.** dN_ch/dη at |η| < 0.5 = 628 (PHOBOS ≈ 609); ⟨p_T⟩ π / K / p
   = 0.48 / 0.72 / 1.04 GeV, a few % harder than the measured central values. Neither causes a factor 2.
7. **Consistent with O+O.** There the model is above sPHENIX by 1–2σ per bin. Au+Au says the same thing
   with much more weight: the current setting loses too little energy.

8. **The p+p reference: a ± 15–25 % systematic on the jet comparison, none on the hadrons** (Figs. 4–5).
   - `pp_ref` describes the measured p+p hadrons as well as Pythia 8 does (weighted model/data): STAR π±
     6–15 GeV 1.12 ± 0.06, PHENIX π⁰ 6–20 GeV 0.86 ± 0.05, STAR h± 6–10 GeV 0.88 ± 0.06; Pythia 8 with the same
     decays: Monash 1.07 / 1.00 / 0.90, Detroit 1.00 / 0.87 / 0.74. The hadron R_AA data use these measured
     references, so the hadron gap is not a reference effect.
   - Charged jets (UE-subtracted, the same decays): `pp_ref` / Detroit 1.01–1.15 (R = 0.2), 0.84–0.96
     (R = 0.4); / Monash 0.83–1.01 (R = 0.2), 0.69–0.84 (R = 0.4). Our jets are somewhat narrower than
     Pythia 8's. Against STAR's measured p+p full jets (R = 0.5) `pp_ref` is right (tuned there) and Pythia 8 is
     10–40 % above (tuning page; full jets hardly depend on the decays).
   - Divided by Pythia 8 as STAR does, the model's R_AA at 16–30 GeV is, for R = 0.2: 0.49–0.58 (Detroit),
     0.40–0.43 (Monash) against 0.46–0.52 with `pp_ref`; for R = 0.4: 0.41–0.57 (Detroit), 0.36–0.44 (Monash)
     against 0.47–0.61. STAR: 0.25–0.38. The reference moves the model by + 10 % to − 25 %, in both
     directions; how STAR's PYTHIA 6 compares with real p+p jets we cannot check here.
   - An earlier version of this point used Pythia 8 with its own decays (weak decays on, π⁰ and η stable), which
     overstated Pythia's charged jets by 5–18 % and the reference effect with them.
   - JETSCAPE's PP19 (their p+p baseline) is within ≈ 10 % of `pp_ref` too (point 9).
   - What is left: jets 1.3–2× STAR at any of the references, hadrons 1.4–1.9× the data. B1 still under-quenches.

9. **JETSCAPE's own chain with the published default reaches the data; ours with Type 2 is about 2× above
   it** (purple steps in Figs. 1–2; 2204.01163 Fig. 14, no retuning at RHIC).
   - JETSCAPE Type 3: charged jets 0.22–0.31 (R = 0.2), 0.19–0.29 (R = 0.4) at 8–30 GeV; pions 0.18–0.28 at
     8–20 GeV. That is STAR's 0.25–0.36 (unbiased) and PHENIX's 0.19–0.38 within ≈ 10–20 %: a little above
     STAR in the biased low-p_T bins (R = 0.4, 8–10 GeV: 0.19 against 0.12), a little below at 25–30 GeV
     (0.28 against 0.36).
   - B1 (our chain, Type 2 α_s 0.25): jets 0.46–0.61, pions ≈ 0.45, i.e. ≈ 2× JETSCAPE's curve at the same
     observables. In Pb+Pb at 5.02 TeV the paper finds Type 2 (0.25) and Type 3 (0.3) equal for jets and Type 2
     *more* suppressing for high-p_T hadrons, so the q̂ form alone is not expected to make a factor 2.
   - B2 (Type 3 α_s 0.3 in our chain) is the direct test: if it lands near JETSCAPE's curve, the setting was the
     cause; if it stays near B1, the difference is in the chain (medium, start time, iMATTER ISR, iColored,
     the p+p baseline).
   - The p+p baseline is not it: JETSCAPE's PP19 (same decays, 160k events) is within ≈ 10 % of `pp_ref`
     (UE-subtracted charged jets pp_ref / PP19 0.85–1.08 at R = 0.2, 0.77–0.94 at R = 0.4; hadrons against data
     PP19 1.03 / 0.96 / 0.82 vs pp_ref 1.12 / 0.86 / 0.88). With PP19 as the reference B1's jet R_AA is 0.42–0.55
     (R = 0.2) and 0.33–0.53 (R = 0.4): still ≈ 1.5–2× JETSCAPE's curve.

**What next:** B2 (Type 3, α_s 0.3, qhatA 10, qhatB 100; same seeds, so the B1 → B2 difference has little
statistical noise) is expected around 14:00–14:30. If B2 is still at ≈ 0.4 or above, raising α_s within
Type 2 (0.35–0.40) is likely not enough on its own; then check why the energy transfer is low in this
chain (the MATTER/LBT handover, Q0, and the temperatures the jets see in our string-sourced hydro) before
scanning further.